In [1]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [2]:
load_dotenv()

True

In [3]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large"
)

In [5]:
database = PineconeVectorStore.from_existing_index(
    index_name="my-tax-index",
    embedding=embeddings
)

In [6]:
recursive = database.as_retriever(
    search_type="similarity",  # 유사도 기반 검색
    search_kwargs={"k": 3}     # 검색할 유사 문서 개수
    )

In [7]:
prompt_template = ChatPromptTemplate.from_template(
    '''
    [Identity]
    당신은 한국의 소득세법 전문가입니다.

    [Answer Rules]
    - 반드시 context로 제공된 내용만 이용해서 사용자의 질문에 친절하게 답변해 주세요.
    - context에 관련 내용이 없다면 "제공된 소득세법 문서에는 관련 내용이 없습니다."라고 답변해 주세요.
    - context에 내용이 없는 경우 추측하지 마세요.
    - 답변 마지막에 [출처]를 작성하세요.
    - 출처는 content에 제공된 문서명과 페이지 번호를 반드시 포함해 주세요.

    [Context]
    {context}

    [Question]
    {query}
    '''
)

In [10]:
def format_response(docs):
    fromated_docs = []
    for idx, doc in enumerate(docs, start=1):
        source = doc.metadata.get('source', '출처 정보 없음')

        fromated_doc = f"""
[검색 문서{idx}]
출처: {source}

내용:
{doc.page_content}
"""
        fromated_docs.append(fromated_doc)

    return "\n".join(fromated_docs)

In [11]:
llm = init_chat_model(
    model="gpt-4o", 
    model_provider='openai',    # "google-genai", "ollama"
    temperature=0
)

In [12]:
rag_chain = (
    {"context": recursive | format_response,  # recursive 결과를 format_response 함수에 전달
     "query": RunnablePassthrough()}          # 전달한 query를 그대로 사용
    | prompt_template
    | llm
    | StrOutputParser()
)

In [20]:
query = "근로소득에 포함되는 소득의 범위에 대해 설명해 주세요."

answer = rag_chain.invoke(query)
print(answer)

근로소득에 포함되는 소득의 범위는 다음과 같습니다:

1. 근로를 제공함으로써 받는 봉급, 급료, 보수, 세비, 임금, 상여, 수당과 이와 유사한 성질의 급여
2. 법인의 주주총회, 사원총회 또는 이에 준하는 의결기관의 결의에 따라 상여로 받는 소득
3. 「법인세법」에 따라 상여로 처분된 금액
4. 퇴직함으로써 받는 소득으로서 퇴직소득에 속하지 아니하는 소득
5. 종업원 등 또는 대학의 교직원이 지급받는 직무발명보상금(제21조제1항제22호의2에 따른 직무발명보상금은 제외)
6. 사업자나 법인이 생산, 공급하는 재화 또는 용역을 그 사업자나 법인의 사업장에 종사하는 임원 등에게 대통령령으로 정하는 바에 따라 시가보다 낮은 가격으로 제공하거나 구입할 수 있도록 지원함으로써 해당 임원 등이 얻는 이익

이러한 소득들은 근로소득으로 간주되며, 근로소득금액은 이러한 소득의 합계액에서 비과세소득을 제외한 금액에 근로소득공제를 적용하여 산출됩니다. [출처: 소득세법_20260701.docx, 제20조, 페이지 번호 미제공]


In [21]:
docs = recursive.invoke(query)
docs

[Document(id='217fbd34-cc76-40af-9899-601b604c9708', metadata={'source': './docs/소득세법_20260701.docx'}, page_content='제20조(근로소득) ① 근로소득은 해당 과세기간에 발생한 다음 각 호의 소득으로 한다. <개정 2016. 12. 20., 2024. 12. 31.>\n\n1. 근로를 제공함으로써 받는 봉급ㆍ급료ㆍ보수ㆍ세비ㆍ임금ㆍ상여ㆍ수당과 이와 유사한 성질의 급여\n\n2. 법인의 주주총회ㆍ사원총회 또는 이에 준하는 의결기관의 결의에 따라 상여로 받는 소득\n\n3. 「법인세법」에 따라 상여로 처분된 금액\n\n4. 퇴직함으로써 받는 소득으로서 퇴직소득에 속하지 아니하는 소득\n\n5. 종업원등 또는 대학의 교직원이 지급받는 직무발명보상금(제21조제1항제22호의2에 따른 직무발명보상금은 제외한다)\n\n6. 사업자나 법인이 생산ㆍ공급하는 재화 또는 용역을 그 사업자나 법인(「독점규제 및 공정거래에 관한 법률」에 따른 계열회사를 포함한다)의 사업장에 종사하는 임원등에게 대통령령으로 정하는 바에 따라 시가보다 낮은 가격으로 제공하거나 구입할 수 있도록 지원함으로써 해당 임원등이 얻는 이익\n\n② 근로소득금액은 제1항 각 호의 소득의 금액의 합계액(비과세소득의 금액은 제외하며, 이하 “총급여액”이라 한다)에서 제47조에 따른 근로소득공제를 적용한 금액으로 한다.\n\n③ 근로소득의 범위에 관하여 필요한 사항은 대통령령으로 정한다.\n\n[전문개정 2009. 12. 31.]\n\n\n\n제20조의2 삭제 <2006. 12. 30.>'),
 Document(id='29d1f5db-d111-41a9-8cb8-f544f162153c', metadata={'source': './docs/소득세법_20260701.docx'}, page_content='제1항 각 호의 소득의 금액의 합계액(비과세소득의 금액은 제외하며, 이하 “총급여액”이라 한다)에서 제47조에 따른 근로소득공제를 적용

In [22]:
context=format_response(docs)
context

'\n[검색 문서1]\n출처: ./docs/소득세법_20260701.docx\n\n내용:\n제20조(근로소득) ① 근로소득은 해당 과세기간에 발생한 다음 각 호의 소득으로 한다. <개정 2016. 12. 20., 2024. 12. 31.>\n\n1. 근로를 제공함으로써 받는 봉급ㆍ급료ㆍ보수ㆍ세비ㆍ임금ㆍ상여ㆍ수당과 이와 유사한 성질의 급여\n\n2. 법인의 주주총회ㆍ사원총회 또는 이에 준하는 의결기관의 결의에 따라 상여로 받는 소득\n\n3. 「법인세법」에 따라 상여로 처분된 금액\n\n4. 퇴직함으로써 받는 소득으로서 퇴직소득에 속하지 아니하는 소득\n\n5. 종업원등 또는 대학의 교직원이 지급받는 직무발명보상금(제21조제1항제22호의2에 따른 직무발명보상금은 제외한다)\n\n6. 사업자나 법인이 생산ㆍ공급하는 재화 또는 용역을 그 사업자나 법인(「독점규제 및 공정거래에 관한 법률」에 따른 계열회사를 포함한다)의 사업장에 종사하는 임원등에게 대통령령으로 정하는 바에 따라 시가보다 낮은 가격으로 제공하거나 구입할 수 있도록 지원함으로써 해당 임원등이 얻는 이익\n\n② 근로소득금액은 제1항 각 호의 소득의 금액의 합계액(비과세소득의 금액은 제외하며, 이하 “총급여액”이라 한다)에서 제47조에 따른 근로소득공제를 적용한 금액으로 한다.\n\n③ 근로소득의 범위에 관하여 필요한 사항은 대통령령으로 정한다.\n\n[전문개정 2009. 12. 31.]\n\n\n\n제20조의2 삭제 <2006. 12. 30.>\n\n\n[검색 문서2]\n출처: ./docs/소득세법_20260701.docx\n\n내용:\n제1항 각 호의 소득의 금액의 합계액(비과세소득의 금액은 제외하며, 이하 “총급여액”이라 한다)에서 제47조에 따른 근로소득공제를 적용한 금액으로 한다.\n\n③ 근로소득의 범위에 관하여 필요한 사항은 대통령령으로 정한다.\n\n[전문개정 2009. 12. 31.]\n\n\n\n제20조의2 삭제 <2006. 12. 30.>\n\n\n\n제20조의3(연금소득) ① 연금소득

In [23]:
prompt_res = prompt_template.invoke({"context": context, "query": query})


print("\n[Prompt 문자열]")
print(prompt_res.to_string())


[Prompt 문자열]
Human: 
    [Identity]
    당신은 한국의 소득세법 전문가입니다.

    [Answer Rules]
    - 반드시 context로 제공된 내용만 이용해서 사용자의 질문에 친절하게 답변해 주세요.
    - context에 관련 내용이 없다면 "제공된 소득세법 문서에는 관련 내용이 없습니다."라고 답변해 주세요.
    - context에 내용이 없는 경우 추측하지 마세요.
    - 답변 마지막에 [출처]를 작성하세요.
    - 출처는 content에 제공된 문서명과 페이지 번호를 반드시 포함해 주세요.

    [Context]
    
[검색 문서1]
출처: ./docs/소득세법_20260701.docx

내용:
제20조(근로소득) ① 근로소득은 해당 과세기간에 발생한 다음 각 호의 소득으로 한다. <개정 2016. 12. 20., 2024. 12. 31.>

1. 근로를 제공함으로써 받는 봉급ㆍ급료ㆍ보수ㆍ세비ㆍ임금ㆍ상여ㆍ수당과 이와 유사한 성질의 급여

2. 법인의 주주총회ㆍ사원총회 또는 이에 준하는 의결기관의 결의에 따라 상여로 받는 소득

3. 「법인세법」에 따라 상여로 처분된 금액

4. 퇴직함으로써 받는 소득으로서 퇴직소득에 속하지 아니하는 소득

5. 종업원등 또는 대학의 교직원이 지급받는 직무발명보상금(제21조제1항제22호의2에 따른 직무발명보상금은 제외한다)

6. 사업자나 법인이 생산ㆍ공급하는 재화 또는 용역을 그 사업자나 법인(「독점규제 및 공정거래에 관한 법률」에 따른 계열회사를 포함한다)의 사업장에 종사하는 임원등에게 대통령령으로 정하는 바에 따라 시가보다 낮은 가격으로 제공하거나 구입할 수 있도록 지원함으로써 해당 임원등이 얻는 이익

② 근로소득금액은 제1항 각 호의 소득의 금액의 합계액(비과세소득의 금액은 제외하며, 이하 “총급여액”이라 한다)에서 제47조에 따른 근로소득공제를 적용한 금액으로 한다.

③ 근로소득의 범위에 관하여 필요한 사항은 대통령령으로 정한다.

[전문개정 

In [24]:
llm_result = llm.invoke(prompt_res)
print(llm_result)

content='근로소득에 포함되는 소득의 범위는 다음과 같습니다:\n\n1. 근로를 제공함으로써 받는 봉급, 급료, 보수, 세비, 임금, 상여, 수당과 이와 유사한 성질의 급여\n2. 법인의 주주총회, 사원총회 또는 이에 준하는 의결기관의 결의에 따라 상여로 받는 소득\n3. 「법인세법」에 따라 상여로 처분된 금액\n4. 퇴직함으로써 받는 소득으로서 퇴직소득에 속하지 아니하는 소득\n5. 종업원 등 또는 대학의 교직원이 지급받는 직무발명보상금(제21조제1항제22호의2에 따른 직무발명보상금은 제외)\n6. 사업자나 법인이 생산, 공급하는 재화 또는 용역을 그 사업자나 법인의 사업장에 종사하는 임원 등에게 대통령령으로 정하는 바에 따라 시가보다 낮은 가격으로 제공하거나 구입할 수 있도록 지원함으로써 해당 임원 등이 얻는 이익\n\n이러한 소득들은 근로소득으로 간주되며, 근로소득금액은 이러한 소득의 합계액에서 비과세소득을 제외한 금액에 근로소득공제를 적용하여 산출됩니다. [출처: 소득세법_20260701.docx, 제20조, 페이지 번호 미제공]' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 331, 'prompt_tokens': 1905, 'total_tokens': 2236, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 1792}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 

In [26]:
parser = StrOutputParser()
parsed_result = parser.parse(llm_result)
print(parsed_result.content)

근로소득에 포함되는 소득의 범위는 다음과 같습니다:

1. 근로를 제공함으로써 받는 봉급, 급료, 보수, 세비, 임금, 상여, 수당과 이와 유사한 성질의 급여
2. 법인의 주주총회, 사원총회 또는 이에 준하는 의결기관의 결의에 따라 상여로 받는 소득
3. 「법인세법」에 따라 상여로 처분된 금액
4. 퇴직함으로써 받는 소득으로서 퇴직소득에 속하지 아니하는 소득
5. 종업원 등 또는 대학의 교직원이 지급받는 직무발명보상금(제21조제1항제22호의2에 따른 직무발명보상금은 제외)
6. 사업자나 법인이 생산, 공급하는 재화 또는 용역을 그 사업자나 법인의 사업장에 종사하는 임원 등에게 대통령령으로 정하는 바에 따라 시가보다 낮은 가격으로 제공하거나 구입할 수 있도록 지원함으로써 해당 임원 등이 얻는 이익

이러한 소득들은 근로소득으로 간주되며, 근로소득금액은 이러한 소득의 합계액에서 비과세소득을 제외한 금액에 근로소득공제를 적용하여 산출됩니다. [출처: 소득세법_20260701.docx, 제20조, 페이지 번호 미제공]
